<img src="https://raw.githubusercontent.com/ComplianceAnalytics/aml-book1/main/assets/cal_logo_banner.png" alt="Compliance Analytics Ltd" width="300" onerror="this.style.display='none'">

# Applied AML Analytics: Turning Data Science Skills into Compliance Decisions
## Chapter 4 — Rule-Based Transaction Monitoring
### Companion Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_04.ipynb)

---

**Book:** *Applied AML Analytics: Turning Data Science Skills into Compliance Decisions* — Book 1  
**Publisher companion repository:** [github.com/ComplianceAnalytics/aml-book1](https://github.com/ComplianceAnalytics/aml-book1)  
**Dataset:** Northgate Retail Bank (synthetic — all data is fictional)  
**Chapters covered:** 3 · 4 · 5 · 6 · 7 · 8 · 9 · **4 (this notebook)**

> **How to use this notebook**  
> Run cells top-to-bottom using **Shift+Enter** or the ▶ button. The setup cell (Section 0) must run first — it generates the Northgate dataset that all later cells depend on. You do not need to install anything; all required libraries are pre-installed in Google Colab.

---

## Contents

| Section | Description | Exercise link |
|---------|-------------|---------------|
| **0. Setup** | Generate the Northgate dataset | — |
| **1. Colab Preview** | Implementing Rule NRB-STRUCT-001: rolling cash structuring (mirrors Section 4.8 of the text) | — |
| **2. Exercise 4.1 Extension** | Threshold sensitivity analysis · false positive profile · alert prioritisation | Exercise 4.1 |
| **3. Reflection cells** | Structured answer prompts | Exercise 4.1 Parts A–C |

---
## Section 0 — Setup: Generate the Northgate Dataset

**Run this cell first.** It generates four CSV files in the Colab session's working directory:

| File | Rows | Description |
|------|------|-------------|
| `nb_transactions.csv` | ~23,000 | All account transactions, Jan–Dec 2023 |
| `nb_customers.csv` | 500 | Customer and account records |
| `nb_counterparties.csv` | 300 | Counterparty firms and their country codes |
| `nb_accounts.csv` | 500 | Account metadata |

The dataset is **fully synthetic**. Northgate Retail Bank does not exist. All account IDs, names, and transactions are generated from a fixed random seed for educational purposes only.

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

rng = np.random.default_rng(42)

# ── Counterparties ────────────────────────────────────────────────────────────
HIGH_RISK  = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']
LOW_RISK   = ['US', 'GB', 'DE', 'FR', 'CA', 'AU', 'SG', 'JP', 'NL', 'CH']

n_cpty      = 300
cpty_ids    = [f'CPT{i:04d}' for i in range(1, n_cpty + 1)]
cpty_cc     = (rng.choice(HIGH_RISK, size=30).tolist() +
               rng.choice(LOW_RISK,  size=270, replace=True).tolist())
rng.shuffle(cpty_cc)
df_cpty = pd.DataFrame({'counterparty_id': cpty_ids, 'country_code': cpty_cc})
df_cpty.to_csv('nb_counterparties.csv', index=False)

n_cust   = 500
cust_ids = [f'NRB_{i:03d}' for i in range(1, n_cust + 1)]
acct_ids = [f'ACC{i:04d}' for i in range(1, n_cust + 1)]
mule_idx = list(range(6))

occupations = ['Employed', 'Self-Employed', 'Retired', 'Student', None]
occ_probs   = [0.55, 0.20, 0.12, 0.08, 0.05]
crr_scores  = rng.choice([1,2,3,4,5], p=[0.35,0.30,0.20,0.10,0.05], size=n_cust)
for i in mule_idx:
    crr_scores[i] = rng.choice([3,4])
incomes_k = rng.lognormal(mean=3.1, sigma=0.5, size=n_cust) * 1000
for i in mule_idx:
    incomes_k[i] = rng.uniform(18, 24) * 1000

df_cust = pd.DataFrame({
    'customer_id':       cust_ids,
    'account_id':        acct_ids,
    'occupation':        rng.choice(occupations, p=occ_probs, size=n_cust),
    'crr_score':         crr_scores,
    'stated_income_usd': np.round(incomes_k, -2),
    'account_open_date': [
        (date(2020,1,1) + timedelta(days=int(d))).isoformat()
        for d in rng.integers(0, 1460, size=n_cust)
    ],
})
df_cust.to_csv('nb_customers.csv', index=False)
df_cust.to_csv('nb_accounts.csv',  index=False)

txn_rows = []
start    = date(2023, 1, 1)
txn_id   = 1

for i, (cid, aid) in enumerate(zip(cust_ids, acct_ids)):
    is_mule = i in mule_idx
    if is_mule:
        for m in range(12):
            for _ in range(rng.integers(3, 9)):
                day      = rng.integers(1, 28)
                txn_date = date(2023, m + 1, day)
                amount   = round(rng.uniform(7800, 9800), 2)
                cpty     = rng.choice(cpty_ids[:30])
                txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                                 'txn_date': txn_date.isoformat(), 'txn_type': 'CASH_IN',
                                 'amount': amount, 'counterparty_id': cpty})
                txn_id += 1
    else:
        for _ in range(rng.integers(12, 80)):
            txn_date = start + timedelta(days=int(rng.integers(0, 365)))
            txn_type = rng.choice(['CASH_IN','TRANSFER_OUT','TRANSFER_IN','CARD'],
                                   p=[0.15, 0.35, 0.35, 0.15])
            amount   = round(min(rng.lognormal(6.5, 1.2), 50000), 2)
            cpty_pool = cpty_ids[30:] if rng.random() > 0.03 else cpty_ids[:30]
            txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                             'txn_date': txn_date.isoformat(), 'txn_type': txn_type,
                             'amount': amount, 'counterparty_id': rng.choice(cpty_pool)})
            txn_id += 1

df_txn = (pd.DataFrame(txn_rows)
            .assign(txn_date=lambda d: pd.to_datetime(d['txn_date']))
            .sort_values('txn_date')
            .reset_index(drop=True))
df_txn.to_csv('nb_transactions.csv', index=False)

print(f"✅ Dataset generated")
print(f"   nb_counterparties : {len(df_cpty):>6,} rows")
print(f"   nb_customers      : {len(df_cust):>6,} rows")
print(f"   nb_transactions   : {len(df_txn):>6,} rows")
print(f"   Date range        : {df_txn['txn_date'].min().date()} → {df_txn['txn_date'].max().date()}")

---
## Section 1 — Colab Preview: Implementing Rule NRB-STRUCT-001

> *This section mirrors Section 4.8 of the textbook exactly. Run the cells and compare the output to the printed figures.*

### What this rule detects

**NRB-STRUCT-001** targets cash structuring: multiple cash deposits, each just below the USD 10,000 reporting threshold, that aggregate to a suspicious total within a 30-day rolling window.

**Rule parameters (default):**
- Transaction type: `CASH_IN` only
- Individual deposit ceiling: `< USD 10,000`
- 30-day rolling window total: `> USD 7,500`
- Minimum deposits in the window: `≥ 3`

These defaults are designed to catch the six known mule accounts embedded in the Northgate dataset. In Chapter 6 you will learn how to tune these thresholds systematically.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the Northgate dataset (generated by Section 0)
df_txn = pd.read_csv('nb_transactions.csv', parse_dates=['txn_date'])

def apply_rule_1(df_txn, threshold=7500, min_txns=3, window_days=30):
    """Rule NRB-STRUCT-001: rolling cash deposits exceeding the threshold."""
    cash = df_txn[
        (df_txn['txn_type'] == 'CASH_IN') &
        (df_txn['amount'] < 10_000)
    ].copy()
    cash = cash.sort_values(['account_id', 'txn_date'])

    results = []
    for acct, grp in cash.groupby('account_id'):
        grp = grp.set_index('txn_date').sort_index()
        rolling_sum = grp['amount'].rolling(f'{window_days}D').sum()
        rolling_cnt = grp['amount'].rolling(f'{window_days}D').count()
        peak_sum = rolling_sum.max()
        peak_cnt = rolling_cnt.max()
        if peak_sum > threshold and peak_cnt >= min_txns:
            results.append({
                'account_id':       acct,
                'peak_rolling_sum': round(peak_sum, 2),
                'peak_txn_count':   int(peak_cnt),
            })
    return pd.DataFrame(results)

alerts = apply_rule_1(df_txn)
print(f"Rule NRB-STRUCT-001 — accounts alerted: {len(alerts)}")
print()
print(alerts.sort_values('peak_rolling_sum', ascending=False).head(10).to_string(index=False))

**What you're seeing:** The rule returns a list of accounts whose rolling 30-day cash deposits exceed USD 7,500 across at least three transactions. All six mule accounts (`ACC0001`–`ACC0006`) should appear near the top of this list, with peak rolling sums in the USD 85,000–110,000 range — far above the threshold.

A handful of non-mule accounts may also appear. This is not a model failure; it reflects the real-world challenge of false positives in rule-based monitoring. In Chapters 6 and 8 you will learn how to measure and reduce them.

In [ ]:
# Visualise: distribution of peak rolling sums for all alerted accounts
MULE_IDS = [f'ACC{i:04d}' for i in range(1, 7)]

fig, ax = plt.subplots(figsize=(7, 4))
non_mule_alerts = alerts[~alerts['account_id'].isin(MULE_IDS)]
mule_alerts     = alerts[alerts['account_id'].isin(MULE_IDS)]

ax.bar(non_mule_alerts['account_id'], non_mule_alerts['peak_rolling_sum'],
       color='#4472C4', alpha=0.7, label='Other alerted accounts')
ax.bar(mule_alerts['account_id'], mule_alerts['peak_rolling_sum'],
       color='#E74C3C', label='Known mule accounts')

ax.axhline(7500, color='black', linestyle='--', linewidth=1, label='Alert threshold (USD 7,500)')
ax.set_xlabel('Account ID', fontsize=10)
ax.set_ylabel('Peak 30-day Rolling Cash Deposit (USD)', fontsize=10)
ax.set_title('Rule NRB-STRUCT-001 — Alerted Accounts\nPeak Rolling Cash Deposit', fontsize=11, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

**Reading the chart:** Mule accounts (red) are clustered at peak rolling sums far above the alert threshold. The few non-mule accounts that trigger the rule (blue) are near the threshold — they are borderline cases rather than clear structuring patterns. This separation matters when you reach Chapter 8: Isolation Forest will use the distance from threshold as one of several features for ML triage.

---
## Section 2 — Exercise 4.1 Extension: Threshold Sensitivity Analysis

> *This section extends Exercise 4.1 from the textbook. The main-text exercise asks you to run `apply_rule_1()` at default settings and review the alert list. This extension invites you to explore how changing the threshold changes alert volume and mule recall.*

**What this means in practice:** Regulators and compliance officers regularly debate where to set rule thresholds. A threshold that is too low generates too many alerts for investigators to review (high false positive rate). A threshold that is too high misses genuine suspicious accounts (low recall). The tension between sensitivity and specificity is central to scenario tuning — which is the topic of Chapter 6.

In [ ]:
# Threshold sensitivity: how does the alert count change across different thresholds?
thresholds = [3000, 5000, 6000, 7500, 9000, 10000, 12000, 15000]
results = []

for t in thresholds:
    a = apply_rule_1(df_txn, threshold=t)
    mule_recall = a['account_id'].isin(MULE_IDS).sum()
    results.append({
        'threshold_usd':  t,
        'total_alerts':   len(a),
        'mule_recall':    mule_recall,
        'false_positives': len(a) - mule_recall,
    })

sensitivity = pd.DataFrame(results)
print("Threshold Sensitivity Analysis — Rule NRB-STRUCT-001")
print(sensitivity.to_string(index=False))

**✏️ YOUR OBSERVATION**

Look at the table above. As the threshold rises from USD 3,000 to USD 15,000:
- At what threshold do you first start losing mule accounts (recall drops below 6)?
- At the default threshold of USD 7,500, how many false positives are there?
- What trade-off would you accept if you had capacity to review only 15 alerts per month?

*Write your answers in the reflection cell in Section 3.*

In [ ]:
# Visualise the trade-off as a chart
fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()

ax1.plot(sensitivity['threshold_usd'], sensitivity['total_alerts'],
         'o-', color='#4472C4', linewidth=2, label='Total alerts (left axis)')
ax2.plot(sensitivity['threshold_usd'], sensitivity['mule_recall'],
         's--', color='#E74C3C', linewidth=2, label='Mule recall (right axis)')

ax1.set_xlabel('Alert Threshold (USD)', fontsize=10)
ax1.set_ylabel('Total Alerted Accounts', fontsize=10, color='#4472C4')
ax2.set_ylabel('Mule Accounts Recalled (out of 6)', fontsize=10, color='#E74C3C')
ax2.set_ylim(0, 8)
ax1.set_title('Rule NRB-STRUCT-001 — Threshold Sensitivity\nAlert Volume vs Mule Recall', fontsize=11, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper right')
ax1.spines['top'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Section 3 — Reflection: Exercise 4.1 Answer Cells

> *Use the cells below to write your answers. Double-click any cell to edit it. When finished, download the notebook (File → Download → Download .ipynb) to submit.*

#### Part A — Understanding the Rule Output

*(Edit this cell to write your answer)*

**How many accounts triggered Rule NRB-STRUCT-001 at the default threshold?**  
  
**Are all six mule accounts present in the alert list? Which account had the highest peak rolling sum?**  
  
**What does the peak transaction count tell you about the structuring behaviour?**  


#### Part B — Threshold Sensitivity

*(Edit this cell to write your answer)*

**At what threshold does the rule first fail to recall all six mule accounts?**  
  
**At the default threshold of USD 7,500, how many accounts are likely false positives? What might explain their presence?**  
  
**If your investigation team can review 20 alerts per month, which threshold would you recommend and why?**  


#### Part C — Limitations of Rule-Based Detection

*(Edit this cell to write your answer)*

**Name two types of structuring behaviour that Rule NRB-STRUCT-001 would NOT detect, even at sensitive threshold settings.**  
  
**In your view, is a rolling 30-day window the right temporal frame for structuring detection? What alternative window lengths might be appropriate?**  


---
## What's Next

In Chapter 5, you will group the 500 Northgate accounts into behavioural segments using K-Means clustering. You will discover that the mule accounts form a distinct cluster — with unusually high cash-in amounts relative to transaction frequency. Segmentation is the foundation for making rule thresholds segment-aware rather than applying the same threshold to every customer.

Open Chapter 6: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_06.ipynb)